# FINANCE 384 Topic 3: Decision Trees

This notebook is the computational companion to Topic 3, *Decision Trees*. It builds the mechanics of tree-based prediction from the same finance perspective used in the lecture slides:

1. calculate impurity and information gain by hand;
2. fit a small classification tree for a lending decision;
3. evaluate default-risk predictions with business error costs;
4. use a bagged-tree ensemble to reduce instability;
5. fit a regression tree for one-month-ahead FF49 industry-portfolio excess returns.

The goal is not to create a deployable credit or trading model. The goal is to learn the workflow discipline: define the decision, respect timing, tune complexity on validation data, hold back the test sample, and interpret tree rules cautiously.


## Learning objectives

After completing this notebook, you should be able to:

1. compute Gini impurity, entropy, weighted post-split impurity, and information gain;
2. explain how a decision tree searches over features and thresholds;
3. fit and interpret a shallow classification tree;
4. evaluate default-risk predictions using confusion-matrix quantities and threshold costs;
5. explain why bagging can stabilize high-variance trees;
6. fit and tune a regression tree for time-ordered return prediction;
7. compare a tree model with a simple benchmark;
8. identify leakage and governance risks in tree-based finance workflows.


## 1. Imports and settings

The core notebook uses NumPy, pandas, and Matplotlib so it can run in the same lightweight environment as the earlier teaching notebooks. The tree functions below are compact teaching implementations. In applied work, analysts usually use a tested library such as `scikit-learn`, XGBoost, or LightGBM.


In [ ]:
import os
import re
import sys
import tempfile
from io import StringIO
from pathlib import Path

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "finance384-matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

CACHE_DIR = Path(tempfile.gettempdir()) / "finance384-cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

if "ipykernel" not in sys.modules and "JPY_PARENT_PID" not in os.environ:
    os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

RANDOM_SEED = 384
rng = np.random.default_rng(RANDOM_SEED)


## 2. Impurity, loss, and tree helpers

The functions in this section implement only the ideas needed for the notebook:

- binary splits of numeric predictors;
- Gini or entropy for classification trees;
- mean squared error for regression trees;
- maximum depth and minimum leaf size as complexity controls;
- optional random feature subsets for bagging and random-forest intuition.

The implementation is intentionally explicit rather than optimized.


In [ ]:
def safe_divide(numerator, denominator):
    return np.nan if denominator == 0 else numerator / denominator


def gini_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    proportions = counts / total
    return 1.0 - np.sum(proportions**2)


def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    proportions = counts[counts > 0] / total
    return -np.sum(proportions * np.log2(proportions))


def class_count_array(series):
    counts = pd.Series(series).value_counts().sort_index()
    return counts.to_numpy(dtype=float)


def classification_impurity(series, criterion="gini"):
    counts = class_count_array(series)
    if criterion == "gini":
        return gini_from_counts(counts)
    if criterion == "entropy":
        return entropy_from_counts(counts)
    raise ValueError("criterion must be 'gini' or 'entropy'")


def mse_impurity(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return 0.0
    return np.mean((values - values.mean()) ** 2)


def candidate_thresholds(values, max_candidates=30):
    values = pd.Series(values).dropna().to_numpy(dtype=float)
    unique_values = np.unique(values)
    if len(unique_values) <= 1:
        return np.array([])

    thresholds = (unique_values[:-1] + unique_values[1:]) / 2.0
    if len(thresholds) > max_candidates:
        probabilities = np.linspace(0.05, 0.95, max_candidates)
        thresholds = np.quantile(values, probabilities)
        thresholds = np.unique(thresholds)
    return thresholds


def choose_feature_subset(feature_cols, max_features=None, rng=None):
    feature_cols = list(feature_cols)
    if max_features is None or len(feature_cols) <= 1:
        return feature_cols

    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)

    if max_features == "sqrt":
        n_features = max(1, int(np.sqrt(len(feature_cols))))
    elif isinstance(max_features, float):
        n_features = max(1, int(np.ceil(max_features * len(feature_cols))))
    else:
        n_features = int(max_features)

    n_features = min(n_features, len(feature_cols))
    return list(rng.choice(feature_cols, size=n_features, replace=False))


In [ ]:
def best_classification_split(
    df,
    feature_cols,
    target_col,
    criterion="gini",
    min_leaf=20,
    max_candidates=30,
    max_features=None,
    rng=None,
):
    parent_impurity = classification_impurity(df[target_col], criterion=criterion)
    features_to_search = choose_feature_subset(feature_cols, max_features=max_features, rng=rng)
    best = None

    for feature in features_to_search:
        for threshold in candidate_thresholds(df[feature], max_candidates=max_candidates):
            left_mask = df[feature] <= threshold
            left_n = int(left_mask.sum())
            right_n = int((~left_mask).sum())
            if left_n < min_leaf or right_n < min_leaf:
                continue

            left_impurity = classification_impurity(df.loc[left_mask, target_col], criterion=criterion)
            right_impurity = classification_impurity(df.loc[~left_mask, target_col], criterion=criterion)
            weighted_impurity = (left_n / len(df)) * left_impurity + (right_n / len(df)) * right_impurity
            gain = parent_impurity - weighted_impurity

            if best is None or weighted_impurity < best["weighted_impurity"]:
                best = {
                    "feature": feature,
                    "threshold": float(threshold),
                    "parent_impurity": float(parent_impurity),
                    "weighted_impurity": float(weighted_impurity),
                    "gain": float(gain),
                    "left_n": left_n,
                    "right_n": right_n,
                }

    return best


def make_classification_leaf(df, target_col):
    probability = float(df[target_col].mean())
    return {
        "type": "leaf",
        "n": int(len(df)),
        "prediction": int(probability >= 0.5),
        "probability": probability,
        "class_counts": df[target_col].value_counts().sort_index().to_dict(),
    }


def build_classification_tree(
    df,
    feature_cols,
    target_col,
    max_depth=3,
    min_leaf=50,
    criterion="gini",
    depth=0,
    max_candidates=30,
    max_features=None,
    rng=None,
):
    if depth >= max_depth or len(df) < 2 * min_leaf or df[target_col].nunique() == 1:
        return make_classification_leaf(df, target_col)

    split = best_classification_split(
        df,
        feature_cols,
        target_col,
        criterion=criterion,
        min_leaf=min_leaf,
        max_candidates=max_candidates,
        max_features=max_features,
        rng=rng,
    )

    if split is None or split["gain"] <= 1e-12:
        return make_classification_leaf(df, target_col)

    left_mask = df[split["feature"]] <= split["threshold"]
    return {
        "type": "node",
        "n": int(len(df)),
        "feature": split["feature"],
        "threshold": split["threshold"],
        "criterion": criterion,
        "impurity": split["parent_impurity"],
        "gain": split["gain"],
        "weighted_impurity": split["weighted_impurity"],
        "left": build_classification_tree(
            df.loc[left_mask],
            feature_cols,
            target_col,
            max_depth=max_depth,
            min_leaf=min_leaf,
            criterion=criterion,
            depth=depth + 1,
            max_candidates=max_candidates,
            max_features=max_features,
            rng=rng,
        ),
        "right": build_classification_tree(
            df.loc[~left_mask],
            feature_cols,
            target_col,
            max_depth=max_depth,
            min_leaf=min_leaf,
            criterion=criterion,
            depth=depth + 1,
            max_candidates=max_candidates,
            max_features=max_features,
            rng=rng,
        ),
    }


def predict_classification_probability_one(tree, row):
    if tree["type"] == "leaf":
        return tree["probability"]
    if row[tree["feature"]] <= tree["threshold"]:
        return predict_classification_probability_one(tree["left"], row)
    return predict_classification_probability_one(tree["right"], row)


def predict_classification_probability(tree, df):
    return np.array([predict_classification_probability_one(tree, row) for _, row in df.iterrows()])


def describe_classification_tree(tree, depth=0):
    indent = "  " * depth
    if tree["type"] == "leaf":
        return [
            (
                f"{indent}Leaf: n={tree['n']}, "
                f"default probability={tree['probability']:.3f}, "
                f"predicted class={tree['prediction']}"
            )
        ]

    lines = [
        (
            f"{indent}if {tree['feature']} <= {tree['threshold']:.4f} "
            f"(n={tree['n']}, gain={tree['gain']:.4f}):"
        )
    ]
    lines.extend(describe_classification_tree(tree["left"], depth + 1))
    lines.append(f"{indent}else:")
    lines.extend(describe_classification_tree(tree["right"], depth + 1))
    return lines


In [ ]:
def best_regression_split(
    df,
    feature_cols,
    target_col,
    min_leaf=100,
    max_candidates=25,
    max_features=None,
    rng=None,
):
    parent_mse = mse_impurity(df[target_col])
    features_to_search = choose_feature_subset(feature_cols, max_features=max_features, rng=rng)
    best = None

    for feature in features_to_search:
        for threshold in candidate_thresholds(df[feature], max_candidates=max_candidates):
            left_mask = df[feature] <= threshold
            left_n = int(left_mask.sum())
            right_n = int((~left_mask).sum())
            if left_n < min_leaf or right_n < min_leaf:
                continue

            left_mse = mse_impurity(df.loc[left_mask, target_col])
            right_mse = mse_impurity(df.loc[~left_mask, target_col])
            weighted_mse = (left_n / len(df)) * left_mse + (right_n / len(df)) * right_mse
            gain = parent_mse - weighted_mse

            if best is None or weighted_mse < best["weighted_mse"]:
                best = {
                    "feature": feature,
                    "threshold": float(threshold),
                    "parent_mse": float(parent_mse),
                    "weighted_mse": float(weighted_mse),
                    "gain": float(gain),
                    "left_n": left_n,
                    "right_n": right_n,
                }

    return best


def make_regression_leaf(df, target_col):
    return {
        "type": "leaf",
        "n": int(len(df)),
        "prediction": float(df[target_col].mean()),
        "target_std": float(df[target_col].std(ddof=0)),
    }


def build_regression_tree(
    df,
    feature_cols,
    target_col,
    max_depth=3,
    min_leaf=200,
    depth=0,
    max_candidates=25,
    max_features=None,
    rng=None,
):
    if depth >= max_depth or len(df) < 2 * min_leaf:
        return make_regression_leaf(df, target_col)

    split = best_regression_split(
        df,
        feature_cols,
        target_col,
        min_leaf=min_leaf,
        max_candidates=max_candidates,
        max_features=max_features,
        rng=rng,
    )

    if split is None or split["gain"] <= 1e-12:
        return make_regression_leaf(df, target_col)

    left_mask = df[split["feature"]] <= split["threshold"]
    return {
        "type": "node",
        "n": int(len(df)),
        "feature": split["feature"],
        "threshold": split["threshold"],
        "mse": split["parent_mse"],
        "gain": split["gain"],
        "weighted_mse": split["weighted_mse"],
        "left": build_regression_tree(
            df.loc[left_mask],
            feature_cols,
            target_col,
            max_depth=max_depth,
            min_leaf=min_leaf,
            depth=depth + 1,
            max_candidates=max_candidates,
            max_features=max_features,
            rng=rng,
        ),
        "right": build_regression_tree(
            df.loc[~left_mask],
            feature_cols,
            target_col,
            max_depth=max_depth,
            min_leaf=min_leaf,
            depth=depth + 1,
            max_candidates=max_candidates,
            max_features=max_features,
            rng=rng,
        ),
    }


def predict_regression_one(tree, row):
    if tree["type"] == "leaf":
        return tree["prediction"]
    if row[tree["feature"]] <= tree["threshold"]:
        return predict_regression_one(tree["left"], row)
    return predict_regression_one(tree["right"], row)


def predict_regression_tree(tree, df):
    return np.array([predict_regression_one(tree, row) for _, row in df.iterrows()])


def describe_regression_tree(tree, depth=0):
    indent = "  " * depth
    if tree["type"] == "leaf":
        return [
            (
                f"{indent}Leaf: n={tree['n']}, "
                f"prediction={tree['prediction']:.4f}, "
                f"within-leaf std={tree['target_std']:.4f}"
            )
        ]

    lines = [
        (
            f"{indent}if {tree['feature']} <= {tree['threshold']:.4f} "
            f"(n={tree['n']}, gain={tree['gain']:.6f}):"
        )
    ]
    lines.extend(describe_regression_tree(tree["left"], depth + 1))
    lines.append(f"{indent}else:")
    lines.extend(describe_regression_tree(tree["right"], depth + 1))
    return lines


def collect_tree_importance(tree, importance=None):
    if importance is None:
        importance = {}
    if tree["type"] == "leaf":
        return importance
    importance[tree["feature"]] = importance.get(tree["feature"], 0.0) + tree["n"] * tree["gain"]
    collect_tree_importance(tree["left"], importance)
    collect_tree_importance(tree["right"], importance)
    return importance


In [ ]:
def regression_metric_table(actual, forecast, label, benchmark_forecast=None):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    error = actual - forecast
    row = {
        "RMSE": np.sqrt(np.mean(error**2)),
        "MAE": np.mean(np.abs(error)),
        "Bias: mean(actual - forecast)": np.mean(error),
    }
    if benchmark_forecast is not None:
        benchmark_error = actual - np.asarray(benchmark_forecast, dtype=float)
        row["OOS_R2_vs_benchmark"] = 1.0 - np.sum(error**2) / np.sum(benchmark_error**2)
    return pd.Series(row, name=label)


def classification_metric_table(actual, probability, threshold, label):
    actual = np.asarray(actual, dtype=int)
    probability = np.asarray(probability, dtype=float)
    predicted_high_risk = (probability >= threshold).astype(int)

    tp = int(((predicted_high_risk == 1) & (actual == 1)).sum())
    fp = int(((predicted_high_risk == 1) & (actual == 0)).sum())
    fn = int(((predicted_high_risk == 0) & (actual == 1)).sum())
    tn = int(((predicted_high_risk == 0) & (actual == 0)).sum())

    return pd.Series(
        {
            "threshold": threshold,
            "accuracy": safe_divide(tp + tn, len(actual)),
            "precision": safe_divide(tp, tp + fp),
            "recall": safe_divide(tp, tp + fn),
            "false_rejection_count": fp,
            "false_approval_count": fn,
            "review_rate": predicted_high_risk.mean(),
            "average_predicted_default_probability": probability.mean(),
            "realized_default_rate": actual.mean(),
            "brier_score": np.mean((actual - probability) ** 2),
        },
        name=label,
    )


def threshold_cost_table(actual, probability, thresholds, false_approval_cost=8.0, false_rejection_cost=1.0):
    rows = []
    actual = np.asarray(actual, dtype=int)
    probability = np.asarray(probability, dtype=float)
    for threshold in thresholds:
        predicted_high_risk = (probability >= threshold).astype(int)
        false_rejection = int(((predicted_high_risk == 1) & (actual == 0)).sum())
        false_approval = int(((predicted_high_risk == 0) & (actual == 1)).sum())
        rows.append(
            {
                "threshold": threshold,
                "review_rate": predicted_high_risk.mean(),
                "defaults_caught_recall": safe_divide(
                    ((predicted_high_risk == 1) & (actual == 1)).sum(),
                    actual.sum(),
                ),
                "false_rejection_count": false_rejection,
                "false_approval_count": false_approval,
                "illustrative_cost": (
                    false_approval_cost * false_approval
                    + false_rejection_cost * false_rejection
                ),
            }
        )
    return pd.DataFrame(rows)


## 3. Manual impurity calculations

Before fitting a tree, compute the split criterion for a small lending example.


In [ ]:
example_counts = pd.DataFrame(
    {
        "node": ["Leaf A", "Leaf B", "Leaf C"],
        "defaults": [12, 40, 0],
        "non_defaults": [68, 40, 50],
    }
)

example_counts["gini"] = example_counts[["defaults", "non_defaults"]].apply(gini_from_counts, axis=1)
example_counts["entropy"] = example_counts[["defaults", "non_defaults"]].apply(entropy_from_counts, axis=1)
example_counts


In [ ]:
parent_gini = gini_from_counts([30, 70])
weighted_child_gini = 0.40 * 0.20 + 0.60 * 0.45
information_gain = parent_gini - weighted_child_gini

pd.Series(
    {
        "parent_gini": parent_gini,
        "weighted_child_gini": weighted_child_gini,
        "information_gain": information_gain,
    }
)


The information gain is positive, so the split makes the child nodes more homogeneous than the parent node. A real tree repeats this search across many features and thresholds.


## 4. Classification tree: synthetic lending decision

This section creates a synthetic loan-application dataset for teaching purposes. It is not evidence about a real lender. The construction makes the timing explicit:

- features are application-time information;
- the target is whether the loan defaults in the next 12 months;
- observations are split chronologically by application month.

The business question is: should a lender flag an application for high-risk review?


In [ ]:
n_loans = 2_400
application_months = pd.date_range("2021-01-31", periods=36, freq=pd.offsets.MonthEnd())

lending = pd.DataFrame(
    {
        "loan_id": np.arange(1, n_loans + 1),
        "application_month": rng.choice(application_months, size=n_loans, replace=True),
        "credit_score": np.clip(rng.normal(690, 62, n_loans), 500, 840),
        "debt_to_income": np.clip(rng.beta(3.2, 5.8, n_loans), 0.05, 0.85),
        "past_delinquencies": rng.poisson(0.45, n_loans),
        "income_verified": rng.binomial(1, 0.76, n_loans),
        "loan_amount": rng.lognormal(mean=10.1, sigma=0.45, size=n_loans),
        "product_type": rng.choice(
            ["auto", "personal", "small_business"],
            size=n_loans,
            p=[0.42, 0.40, 0.18],
        ),
    }
)

lending["loan_amount_log"] = np.log(lending["loan_amount"])
lending["small_business"] = (lending["product_type"] == "small_business").astype(int)
lending["personal"] = (lending["product_type"] == "personal").astype(int)

latent_default_score = (
    -3.9
    + 0.010 * (660 - lending["credit_score"])
    + 3.0 * (lending["debt_to_income"] - 0.34)
    + 0.42 * lending["past_delinquencies"]
    + 0.58 * (1 - lending["income_verified"])
    + 0.35 * lending["small_business"]
    + 0.22 * lending["personal"]
    + 0.85 * (
        (lending["debt_to_income"] > 0.48)
        & (lending["credit_score"] < 645)
    ).astype(int)
)

default_probability = 1.0 / (1.0 + np.exp(-latent_default_score))
lending["default_next_12m"] = rng.binomial(1, default_probability)
lending = lending.sort_values(["application_month", "loan_id"]).reset_index(drop=True)

lending.head()


In [ ]:
feature_cols_lending = [
    "credit_score",
    "debt_to_income",
    "past_delinquencies",
    "income_verified",
    "loan_amount_log",
    "personal",
    "small_business",
]
target_lending = "default_next_12m"

required_columns = set(["loan_id", "application_month", target_lending] + feature_cols_lending)
missing_columns = sorted(required_columns - set(lending.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if lending["loan_id"].duplicated().any():
    raise ValueError("loan_id should be unique in this teaching dataset.")

if lending[feature_cols_lending + [target_lending]].isna().sum().sum() != 0:
    raise ValueError("Missing values are present after feature construction.")

summary = pd.Series(
    {
        "rows": len(lending),
        "application_months": lending["application_month"].nunique(),
        "default_rate": lending[target_lending].mean(),
        "first_month": lending["application_month"].min().strftime("%Y-%m"),
        "last_month": lending["application_month"].max().strftime("%Y-%m"),
    }
)
summary


The product-type variable has been encoded as numeric indicator variables (`personal` and `small_business`). This is not just a coding detail. Many tree APIs expect a numeric design matrix, and different libraries handle categorical variables and missing values differently. A finance analyst should make these choices explicit before fitting the model.


### Chronological split

The validation and test samples come from later application months. This prevents a model from being tuned using future default outcomes.


In [ ]:
unique_application_months = pd.Index(sorted(lending["application_month"].unique()))
train_months = unique_application_months[:24]
valid_months = unique_application_months[24:30]
test_months = unique_application_months[30:]

lend_train = lending.loc[lending["application_month"].isin(train_months)].copy()
lend_valid = lending.loc[lending["application_month"].isin(valid_months)].copy()
lend_test = lending.loc[lending["application_month"].isin(test_months)].copy()

pd.DataFrame(
    {
        "sample": ["train", "validation", "test"],
        "rows": [len(lend_train), len(lend_valid), len(lend_test)],
        "first_month": [
            lend_train["application_month"].min().strftime("%Y-%m"),
            lend_valid["application_month"].min().strftime("%Y-%m"),
            lend_test["application_month"].min().strftime("%Y-%m"),
        ],
        "last_month": [
            lend_train["application_month"].max().strftime("%Y-%m"),
            lend_valid["application_month"].max().strftime("%Y-%m"),
            lend_test["application_month"].max().strftime("%Y-%m"),
        ],
        "default_rate": [
            lend_train[target_lending].mean(),
            lend_valid[target_lending].mean(),
            lend_test[target_lending].mean(),
        ],
    }
)


### Tune tree depth and leaf size

The tree is fitted on the training sample. Complexity is selected on the validation sample using the Brier score, which measures the squared error in predicted probabilities.


In [ ]:
classification_tuning_rows = []
best_classification_model = None

for max_depth in [1, 2, 3, 4]:
    for min_leaf in [40, 80, 120]:
        tree = build_classification_tree(
            lend_train,
            feature_cols_lending,
            target_lending,
            max_depth=max_depth,
            min_leaf=min_leaf,
            criterion="gini",
            max_candidates=25,
        )
        valid_probability = predict_classification_probability(tree, lend_valid)
        valid_brier = np.mean((lend_valid[target_lending].to_numpy() - valid_probability) ** 2)
        classification_tuning_rows.append(
            {
                "max_depth": max_depth,
                "min_leaf": min_leaf,
                "validation_brier": valid_brier,
                "validation_average_probability": valid_probability.mean(),
            }
        )
        if best_classification_model is None or valid_brier < best_classification_model["validation_brier"]:
            best_classification_model = {
                "tree": tree,
                "max_depth": max_depth,
                "min_leaf": min_leaf,
                "validation_brier": valid_brier,
            }

classification_tuning = pd.DataFrame(classification_tuning_rows).sort_values("validation_brier")
classification_tuning.head(8)


In [ ]:
print(
    f"Selected max_depth={best_classification_model['max_depth']} "
    f"and min_leaf={best_classification_model['min_leaf']}."
)
print()
for line in describe_classification_tree(best_classification_model["tree"]):
    print(line)


The printed `predicted class` uses the default majority-class rule with a 0.5 cutoff. In credit-risk review, a lender will often use a much lower probability threshold because default is rare and costly. The next step chooses that action threshold on validation data.


### Choose a decision threshold

Predicted default probabilities must be translated into an action. The table below uses illustrative cost units:

- false approval: predicted low risk, but the borrower defaults;
- false rejection or unnecessary review: predicted high risk, but the borrower does not default.

The false-approval cost is set higher because approving a defaulting loan is usually more costly than sending a good loan to review. These cost weights are teaching assumptions, not bank policy.


In [ ]:
valid_probability_tree = predict_classification_probability(best_classification_model["tree"], lend_valid)
test_probability_tree = predict_classification_probability(best_classification_model["tree"], lend_test)

thresholds = np.arange(0.05, 0.51, 0.05)
valid_threshold_costs = threshold_cost_table(
    lend_valid[target_lending],
    valid_probability_tree,
    thresholds,
    false_approval_cost=8.0,
    false_rejection_cost=1.0,
)

selected_threshold = valid_threshold_costs.loc[
    valid_threshold_costs["illustrative_cost"].idxmin(),
    "threshold",
]

valid_threshold_costs


In [ ]:
single_tree_metrics = pd.concat(
    [
        classification_metric_table(
            lend_train[target_lending],
            predict_classification_probability(best_classification_model["tree"], lend_train),
            selected_threshold,
            "single tree train",
        ),
        classification_metric_table(
            lend_valid[target_lending],
            valid_probability_tree,
            selected_threshold,
            "single tree validation",
        ),
        classification_metric_table(
            lend_test[target_lending],
            test_probability_tree,
            selected_threshold,
            "single tree test",
        ),
    ],
    axis=1,
).T

single_tree_metrics


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
valid_threshold_costs.plot(
    x="threshold",
    y=["false_approval_count", "false_rejection_count"],
    marker="o",
    ax=ax,
)
ax.axvline(selected_threshold, color="#003865", linestyle="--", linewidth=1.5)
ax.set_title("Validation threshold tradeoff")
ax.set_ylabel("Count")
ax.set_xlabel("High-risk threshold")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## 5. Bagged trees: reducing instability

A single tree can be sensitive to the training sample. Bagging fits many trees to bootstrap samples and averages their predicted probabilities. The implementation below also uses a random subset of features at each split, which gives the intuition of a random forest.


In [ ]:
def fit_bagged_classification_trees(
    train_df,
    feature_cols,
    target_col,
    n_trees=40,
    max_depth=4,
    min_leaf=70,
    max_candidates=20,
    max_features="sqrt",
    random_seed=RANDOM_SEED,
):
    rng_local = np.random.default_rng(random_seed)
    trees = []
    for _ in range(n_trees):
        sample_positions = rng_local.integers(0, len(train_df), size=len(train_df))
        bootstrap_sample = train_df.iloc[sample_positions].copy()
        tree = build_classification_tree(
            bootstrap_sample,
            feature_cols,
            target_col,
            max_depth=max_depth,
            min_leaf=min_leaf,
            criterion="gini",
            max_candidates=max_candidates,
            max_features=max_features,
            rng=rng_local,
        )
        trees.append(tree)
    return trees


def predict_bagged_classification_probability(trees, df):
    probabilities = np.column_stack([
        predict_classification_probability(tree, df)
        for tree in trees
    ])
    return probabilities.mean(axis=1)


bagged_trees = fit_bagged_classification_trees(
    lend_train,
    feature_cols_lending,
    target_lending,
    n_trees=40,
    max_depth=4,
    min_leaf=70,
    max_candidates=20,
    max_features="sqrt",
)

bagged_valid_probability = predict_bagged_classification_probability(bagged_trees, lend_valid)
bagged_test_probability = predict_bagged_classification_probability(bagged_trees, lend_test)

bagged_threshold_costs = threshold_cost_table(
    lend_valid[target_lending],
    bagged_valid_probability,
    thresholds,
    false_approval_cost=8.0,
    false_rejection_cost=1.0,
)
bagged_threshold = bagged_threshold_costs.loc[
    bagged_threshold_costs["illustrative_cost"].idxmin(),
    "threshold",
]

bagged_threshold_costs


In [ ]:
bagged_metrics = pd.concat(
    [
        classification_metric_table(
            lend_valid[target_lending],
            bagged_valid_probability,
            bagged_threshold,
            "bagged trees validation",
        ),
        classification_metric_table(
            lend_test[target_lending],
            bagged_test_probability,
            bagged_threshold,
            "bagged trees test",
        ),
    ],
    axis=1,
).T

pd.concat([single_tree_metrics.loc[["single tree validation", "single tree test"]], bagged_metrics])


The bagged model is less directly interpretable than the single tree, but averaging can improve probability stability. Whether the tradeoff is worthwhile depends on the decision, the size of the performance gain, and the explanation obligations attached to the model.


## 6. Regression tree: FF49 one-month-ahead excess returns

This section uses the local Kenneth French industry-portfolio files from Topic 1:

- `49_Industry_Portfolios.csv`;
- `F-F_Research_Data_Factors.csv`.

The target is next month's industry-portfolio excess return. Predictors are known by the end of the current month: lagged portfolio returns, rolling return summaries, and current Fama-French factor returns.

This is a teaching exercise about model evaluation, not a trading recommendation.


In [ ]:
INDUSTRY_PORTFOLIOS_CANDIDATES = [
    Path("Topic1_PooledData/4.Code/data/49_Industry_Portfolios.csv"),
    Path("../Topic1_PooledData/4.Code/data/49_Industry_Portfolios.csv"),
    Path("../../Topic1_PooledData/4.Code/data/49_Industry_Portfolios.csv"),
    Path("data/49_Industry_Portfolios.csv"),
]

FACTOR_CANDIDATES = [
    Path("Topic1_PooledData/4.Code/data/F-F_Research_Data_Factors.csv"),
    Path("../Topic1_PooledData/4.Code/data/F-F_Research_Data_Factors.csv"),
    Path("../../Topic1_PooledData/4.Code/data/F-F_Research_Data_Factors.csv"),
    Path("data/F-F_Research_Data_Factors.csv"),
]

MISSING_RETURN_CODES = [-99.99, -999.0]


def find_first_existing_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    searched = "\n".join(str(path.resolve()) for path in candidates)
    raise FileNotFoundError("Could not find required data file. Searched:\n" + searched)


def monthly_table_from_ken_french_csv(path, section_title=None):
    lines = Path(path).read_text(encoding="utf-8-sig").splitlines()

    if section_title is None:
        header_idx = next(
            i for i, line in enumerate(lines)
            if line.strip().startswith(",") and "RF" in line
        )
    else:
        section_idx = next(
            i for i, line in enumerate(lines)
            if line.strip() == section_title
        )
        header_idx = next(
            i for i in range(section_idx + 1, len(lines))
            if lines[i].strip()
        )

    rows = [lines[header_idx]]
    for line in lines[header_idx + 1:]:
        if not re.match(r"^\s*\d{6}\s*,", line):
            break
        rows.append(line)

    table = pd.read_csv(StringIO("\n".join(rows)))
    table = table.rename(columns={table.columns[0]: "yyyymm"})
    table["date"] = pd.PeriodIndex(table["yyyymm"].astype(str), freq="M").to_timestamp("M")
    return table.drop(columns="yyyymm")


industry_path = find_first_existing_path(INDUSTRY_PORTFOLIOS_CANDIDATES)
factor_path = find_first_existing_path(FACTOR_CANDIDATES)

industry_wide = monthly_table_from_ken_french_csv(
    industry_path,
    section_title="Average Value Weighted Returns -- Monthly",
)
factors = monthly_table_from_ken_french_csv(factor_path)

print(f"Industry portfolio file: {industry_path.resolve()}")
print(f"Factor file: {factor_path.resolve()}")
print(f"Industry rows: {len(industry_wide):,}")
print(f"Factor rows: {len(factors):,}")
industry_wide.head()


In [ ]:
industry_cols = [col for col in industry_wide.columns if col != "date"]
industry_name_map = {col: col.strip() for col in industry_cols}
industry_wide = industry_wide.rename(columns=industry_name_map)
industry_cols = [industry_name_map[col] for col in industry_cols]

for col in industry_cols:
    industry_wide[col] = pd.to_numeric(industry_wide[col], errors="coerce")
    industry_wide[col] = industry_wide[col].replace(MISSING_RETURN_CODES, np.nan) / 100.0

portfolio_panel = industry_wide.melt(
    id_vars="date",
    value_vars=industry_cols,
    var_name="ff49_name",
    value_name="portfolio_ret",
).dropna(subset=["portfolio_ret"]).copy()

industry_codes = {name: code for code, name in enumerate(industry_cols, start=1)}
portfolio_panel["ff49_code"] = portfolio_panel["ff49_name"].map(industry_codes)
portfolio_panel["portfolio"] = (
    portfolio_panel["ff49_code"].astype(int).astype(str).str.zfill(2)
    + "_"
    + portfolio_panel["ff49_name"]
)

factors = factors.rename(columns={
    "Mkt-RF": "mktrf",
    "SMB": "smb",
    "HML": "hml",
    "RF": "rf",
})
for col in ["mktrf", "smb", "hml", "rf"]:
    factors[col] = pd.to_numeric(factors[col], errors="coerce") / 100.0

portfolio_panel = portfolio_panel.merge(
    factors[["date", "mktrf", "smb", "hml", "rf"]],
    how="inner",
    on="date",
    validate="many_to_one",
)
portfolio_panel["portfolio_excess_ret"] = portfolio_panel["portfolio_ret"] - portfolio_panel["rf"]
portfolio_panel = portfolio_panel.dropna(subset=[
    "portfolio_excess_ret", "mktrf", "smb", "hml", "rf", "portfolio"
]).copy()
portfolio_panel = portfolio_panel.sort_values(["portfolio", "date"]).reset_index(drop=True)

portfolio_panel.head()


### Feature construction with timing

For each portfolio, the target is next month's excess return. Lagged portfolio features use only information available by the end of the current month.


In [ ]:
panel = portfolio_panel.copy()
grouped = panel.groupby("portfolio", group_keys=False)

panel["excess_lag1"] = grouped["portfolio_excess_ret"].shift(1)
panel["excess_lag2"] = grouped["portfolio_excess_ret"].shift(2)
panel["excess_lag3"] = grouped["portfolio_excess_ret"].shift(3)
panel["excess_mean_3m"] = grouped["portfolio_excess_ret"].transform(
    lambda s: s.shift(1).rolling(3).mean()
)
panel["excess_vol_12m"] = grouped["portfolio_excess_ret"].transform(
    lambda s: s.shift(1).rolling(12).std(ddof=0)
)
panel["target_next_excess_ret"] = grouped["portfolio_excess_ret"].shift(-1)

factor_time = factors[["date", "mktrf", "smb", "hml"]].copy().sort_values("date")
factor_time["mktrf_mean_3m"] = factor_time["mktrf"].rolling(3).mean()
factor_time["mktrf_vol_12m"] = factor_time["mktrf"].rolling(12).std(ddof=0)

panel = panel.drop(columns=["mktrf", "smb", "hml"]).merge(
    factor_time,
    on="date",
    how="left",
    validate="many_to_one",
)

feature_cols_returns = [
    "excess_lag1",
    "excess_lag2",
    "excess_lag3",
    "excess_mean_3m",
    "excess_vol_12m",
    "mktrf",
    "smb",
    "hml",
    "mktrf_mean_3m",
    "mktrf_vol_12m",
]
target_returns = "target_next_excess_ret"

model_returns = panel.dropna(subset=feature_cols_returns + [target_returns]).copy()
model_returns = model_returns.loc[model_returns["date"] >= pd.Timestamp("1963-07-31")].copy()
model_returns = model_returns.sort_values(["date", "portfolio"]).reset_index(drop=True)

if model_returns.duplicated(["portfolio", "date"]).any():
    raise ValueError("Portfolio-date observations should be unique.")

pd.Series(
    {
        "rows": len(model_returns),
        "portfolios": model_returns["portfolio"].nunique(),
        "months": model_returns["date"].nunique(),
        "first_month": model_returns["date"].min().strftime("%Y-%m"),
        "last_month": model_returns["date"].max().strftime("%Y-%m"),
    }
)


In [ ]:
model_returns[["date", "portfolio", target_returns] + feature_cols_returns[:6]].head()


### Time-ordered train, validation, and test samples

We use the first 60 percent of months for training, the next 20 percent for validation, and the final 20 percent for testing.


In [ ]:
unique_return_months = pd.Index(sorted(model_returns["date"].unique()))
train_end = int(0.60 * len(unique_return_months))
valid_end = int(0.80 * len(unique_return_months))

return_train_months = unique_return_months[:train_end]
return_valid_months = unique_return_months[train_end:valid_end]
return_test_months = unique_return_months[valid_end:]

return_train = model_returns.loc[model_returns["date"].isin(return_train_months)].copy()
return_valid = model_returns.loc[model_returns["date"].isin(return_valid_months)].copy()
return_test = model_returns.loc[model_returns["date"].isin(return_test_months)].copy()

pd.DataFrame(
    {
        "sample": ["train", "validation", "test"],
        "rows": [len(return_train), len(return_valid), len(return_test)],
        "first_month": [
            return_train["date"].min().strftime("%Y-%m"),
            return_valid["date"].min().strftime("%Y-%m"),
            return_test["date"].min().strftime("%Y-%m"),
        ],
        "last_month": [
            return_train["date"].max().strftime("%Y-%m"),
            return_valid["date"].max().strftime("%Y-%m"),
            return_test["date"].max().strftime("%Y-%m"),
        ],
        "mean_next_excess_return": [
            return_train[target_returns].mean(),
            return_valid[target_returns].mean(),
            return_test[target_returns].mean(),
        ],
    }
)


### Tune a regression tree

We tune maximum depth and minimum leaf size on validation RMSE. The test sample is not used for model selection.


In [ ]:
regression_tuning_rows = []
best_regression_model = None

for max_depth in [1, 2, 3, 4]:
    for min_leaf in [250, 500, 750]:
        tree = build_regression_tree(
            return_train,
            feature_cols_returns,
            target_returns,
            max_depth=max_depth,
            min_leaf=min_leaf,
            max_candidates=18,
        )
        valid_forecast = predict_regression_tree(tree, return_valid)
        valid_rmse = regression_metric_table(
            return_valid[target_returns],
            valid_forecast,
            "validation",
        )["RMSE"]
        regression_tuning_rows.append(
            {
                "max_depth": max_depth,
                "min_leaf": min_leaf,
                "validation_RMSE": valid_rmse,
            }
        )
        if best_regression_model is None or valid_rmse < best_regression_model["validation_RMSE"]:
            best_regression_model = {
                "tree": tree,
                "max_depth": max_depth,
                "min_leaf": min_leaf,
                "validation_RMSE": valid_rmse,
            }

regression_tuning = pd.DataFrame(regression_tuning_rows).sort_values("validation_RMSE")
regression_tuning


In [ ]:
print(
    f"Selected max_depth={best_regression_model['max_depth']} "
    f"and min_leaf={best_regression_model['min_leaf']}."
)
print()
for line in describe_regression_tree(best_regression_model["tree"]):
    print(line)


### Compare with a historical-mean benchmark

The benchmark forecast is the average next-month excess return in the training sample. This is simple, transparent, and difficult to beat in noisy return data.


In [ ]:
historical_mean_forecast = return_train[target_returns].mean()

train_tree_forecast = predict_regression_tree(best_regression_model["tree"], return_train)
valid_tree_forecast = predict_regression_tree(best_regression_model["tree"], return_valid)
test_tree_forecast = predict_regression_tree(best_regression_model["tree"], return_test)

return_metrics = pd.concat(
    [
        regression_metric_table(
            return_train[target_returns],
            np.full(len(return_train), historical_mean_forecast),
            "historical mean train",
        ),
        regression_metric_table(
            return_valid[target_returns],
            np.full(len(return_valid), historical_mean_forecast),
            "historical mean validation",
        ),
        regression_metric_table(
            return_test[target_returns],
            np.full(len(return_test), historical_mean_forecast),
            "historical mean test",
        ),
        regression_metric_table(
            return_train[target_returns],
            train_tree_forecast,
            "tree train",
            benchmark_forecast=np.full(len(return_train), historical_mean_forecast),
        ),
        regression_metric_table(
            return_valid[target_returns],
            valid_tree_forecast,
            "tree validation",
            benchmark_forecast=np.full(len(return_valid), historical_mean_forecast),
        ),
        regression_metric_table(
            return_test[target_returns],
            test_tree_forecast,
            "tree test",
            benchmark_forecast=np.full(len(return_test), historical_mean_forecast),
        ),
    ],
    axis=1,
).T

return_metrics


In this fixed run, the selected tree improves validation performance only slightly and does not beat the historical-mean benchmark on the final test sample. That is a useful result, not a failure of the exercise. It illustrates why return-prediction models must be evaluated out of sample and why an economically plausible split is not enough to justify a trading strategy.


In [ ]:
importance = collect_tree_importance(best_regression_model["tree"])
importance_table = (
    pd.Series(importance, name="weighted_impurity_reduction")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "feature"})
)
importance_table["share"] = (
    importance_table["weighted_impurity_reduction"]
    / importance_table["weighted_impurity_reduction"].sum()
)
importance_table


In [ ]:
test_plot = return_test[["date", target_returns]].copy()
test_plot["tree_forecast"] = test_tree_forecast
test_by_month = test_plot.groupby("date")[[target_returns, "tree_forecast"]].mean()

fig, ax = plt.subplots(figsize=(10, 4))
test_by_month.plot(ax=ax, linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Test-sample average next-month excess return and tree forecast")
ax.set_ylabel("Monthly excess return")
ax.set_xlabel("Forecast month")
ax.legend(["Realized average", "Tree forecast"], frameon=False)
plt.tight_layout()
plt.show()


Return prediction is noisy. A tree that looks economically intuitive can still fail to improve out of sample. The correct conclusion should be based on the validation and test evidence, not on the attractiveness of a particular split.


## 7. Optional: compare with the scikit-learn API

The course generally uses `scikit-learn` for machine-learning workflows. The local environment used to build these materials may not have it installed, so this cell skips gracefully if the package is unavailable.


In [ ]:
try:
    from sklearn.tree import DecisionTreeClassifier

    sklearn_tree = DecisionTreeClassifier(
        max_depth=best_classification_model["max_depth"],
        min_samples_leaf=best_classification_model["min_leaf"],
        criterion="gini",
        random_state=RANDOM_SEED,
    )
    sklearn_tree.fit(lend_train[feature_cols_lending], lend_train[target_lending])
    sklearn_valid_probability = sklearn_tree.predict_proba(lend_valid[feature_cols_lending])[:, 1]
    sklearn_metric = classification_metric_table(
        lend_valid[target_lending],
        sklearn_valid_probability,
        selected_threshold,
        "scikit-learn validation",
    )
    sklearn_metric
except ModuleNotFoundError:
    print("scikit-learn is not installed. The notebook core uses the teaching implementation above.")


## 8. Governance checklist

Before using a tree-based model in a financial decision, answer these questions:

1. Are all predictors observable before the target is realized?
2. Does the train-validation-test design match the deployment timeline?
3. Is there a simple benchmark, and does the tree improve on it out of sample?
4. Are the most important splits economically sensible?
5. Are errors concentrated in particular borrower, firm, or market segments?
6. Could a feature act as a proxy for a protected or sensitive attribute?
7. Can the institution explain adverse decisions to affected stakeholders?
8. How will the model be monitored for drift, calibration decay, and regime change?
